# PyTorch Models and DataLoaders

Goal: move from manually defined parameters to a real PyTorch model and train it using mini-batches.

In [1]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

x = torch.arange(1, 101, dtype=torch.float32).reshape(-1, 1)
y = 3 * x + 2

print("x shape:", x.shape)
print("y shape:", y.shape)

x shape: torch.Size([100, 1])
y shape: torch.Size([100, 1])


In [2]:
# define model

class LinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)

    def forward(self, x):
        return self.linear(x)

In [3]:
# instantiate model

model = LinearModel() # similar to "predictions = x @ w + b" but now pytorch owns and manages w and b

print(model)

LinearModel(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)


In [4]:
# model inspection
for name, param in model.named_parameters():
    print(name, param.data)

linear.weight tensor([[0.8150]])
linear.bias tensor([0.5163])


In [5]:
# dataset creation
dataset = TensorDataset(x, y)

In [6]:
# dataloader creation to load batches of 10 at a time, 
# shuffling the data each epoch

loader = DataLoader(
    dataset, 
    batch_size=10, 
    shuffle=True
)

In [7]:
# single batch inspection

xb, yb = next(iter(loader))

print("xb shape:", xb.shape)
print("yb shape:", yb.shape)

print("\nxb:\n", xb)
print("\nyb:\n", yb)

xb shape: torch.Size([10, 1])
yb shape: torch.Size([10, 1])

xb:
 tensor([[ 26.],
        [ 48.],
        [ 90.],
        [100.],
        [ 46.],
        [  8.],
        [ 66.],
        [ 14.],
        [ 83.],
        [ 16.]])

yb:
 tensor([[ 80.],
        [146.],
        [272.],
        [302.],
        [140.],
        [ 26.],
        [200.],
        [ 44.],
        [251.],
        [ 50.]])


In [8]:
criterion = nn.MSELoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.0001
)

In [9]:
epochs = 2500

for epoch in range(epochs):

    total_loss = 0.0

    for xb, yb in loader:

        optimizer.zero_grad()

        predictions = model(xb)

        loss = criterion(predictions, yb)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)

    if epoch % 5 == 0:
        print(
            f"Epoch {epoch:3d} | "
            f"Loss: {avg_loss:.4f}"
        )

Epoch   0 | Loss: 2147.2562
Epoch   5 | Loss: 0.5266
Epoch  10 | Loss: 0.5326
Epoch  15 | Loss: 0.5148
Epoch  20 | Loss: 0.5163
Epoch  25 | Loss: 0.5221
Epoch  30 | Loss: 0.5104
Epoch  35 | Loss: 0.5096
Epoch  40 | Loss: 0.5139
Epoch  45 | Loss: 0.5076
Epoch  50 | Loss: 0.5172
Epoch  55 | Loss: 0.5034
Epoch  60 | Loss: 0.4937
Epoch  65 | Loss: 0.4941
Epoch  70 | Loss: 0.4919
Epoch  75 | Loss: 0.4993
Epoch  80 | Loss: 0.5029
Epoch  85 | Loss: 0.4840
Epoch  90 | Loss: 0.4916
Epoch  95 | Loss: 0.4893
Epoch 100 | Loss: 0.4829
Epoch 105 | Loss: 0.4949
Epoch 110 | Loss: 0.4717
Epoch 115 | Loss: 0.4672
Epoch 120 | Loss: 0.4661
Epoch 125 | Loss: 0.4639
Epoch 130 | Loss: 0.4622
Epoch 135 | Loss: 0.4645
Epoch 140 | Loss: 0.4653
Epoch 145 | Loss: 0.4504
Epoch 150 | Loss: 0.4546
Epoch 155 | Loss: 0.4547
Epoch 160 | Loss: 0.4561
Epoch 165 | Loss: 0.4519
Epoch 170 | Loss: 0.4433
Epoch 175 | Loss: 0.4370
Epoch 180 | Loss: 0.4475
Epoch 185 | Loss: 0.4458
Epoch 190 | Loss: 0.4615
Epoch 195 | Loss: 0.41

In [10]:
for name, param in model.named_parameters():
    print(name, param.data)

linear.weight tensor([[3.0047]])
linear.bias tensor([1.5778])


## Train/Validation Split and GPU Training

In [11]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, random_split

x = torch.arange(1, 101, dtype=torch.float32).reshape(-1, 1)
y = 3 * x + 2

In [12]:
dataset = TensorDataset(x, y)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

Train size: 80
Validation size: 20


In [13]:
train_loader = DataLoader(
    train_dataset,
    batch_size=10,
    shuffle=True # shuffle training data each epoch
)

val_loader = DataLoader(
    val_dataset,
    batch_size=10,
    shuffle=False # no need to shuffle validation data
)

In [14]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu" # use GPU if available, else CPU
)

print("Using:", device)

Using: cuda


In [15]:
class LinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)

    def forward(self, x):
        return self.linear(x)

In [16]:
model = LinearModel().to(device) # move model to the device from cpu to gpu

print(model)
print("Model device:", next(model.parameters()).device)

LinearModel(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)
Model device: cuda:0


In [17]:
# define loss function and optimizer

criterion = nn.MSELoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.0001
)

In [18]:
# training loop

epochs = 2500

for epoch in range(epochs):

    model.train()
    train_loss = 0.0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        predictions = model(xb)

        loss = criterion(predictions, yb)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            predictions = model(xb)

            loss = criterion(predictions, yb)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    if epoch % 5 == 0:
        print(
            f"Epoch {epoch:3d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f}"
        )

Epoch   0 | Train Loss: 2535.7068 | Val Loss: 1.9241
Epoch   5 | Train Loss: 1.5794 | Val Loss: 1.9029
Epoch  10 | Train Loss: 1.6421 | Val Loss: 1.8940
Epoch  15 | Train Loss: 1.5318 | Val Loss: 1.8940
Epoch  20 | Train Loss: 1.5872 | Val Loss: 1.8939
Epoch  25 | Train Loss: 1.6167 | Val Loss: 1.9377
Epoch  30 | Train Loss: 1.6678 | Val Loss: 1.9444
Epoch  35 | Train Loss: 1.5329 | Val Loss: 1.8520
Epoch  40 | Train Loss: 1.5324 | Val Loss: 1.8662
Epoch  45 | Train Loss: 1.5188 | Val Loss: 1.8488
Epoch  50 | Train Loss: 1.5260 | Val Loss: 1.9058
Epoch  55 | Train Loss: 1.5215 | Val Loss: 1.8437
Epoch  60 | Train Loss: 1.5763 | Val Loss: 1.8388
Epoch  65 | Train Loss: 1.5190 | Val Loss: 1.8427
Epoch  70 | Train Loss: 1.5463 | Val Loss: 1.8164
Epoch  75 | Train Loss: 1.5167 | Val Loss: 1.7953
Epoch  80 | Train Loss: 1.4683 | Val Loss: 1.7889
Epoch  85 | Train Loss: 1.4613 | Val Loss: 1.8003
Epoch  90 | Train Loss: 1.5276 | Val Loss: 1.8359
Epoch  95 | Train Loss: 1.4812 | Val Loss: 1.76

In [19]:
# model.train() tells PyTorch the model is in training mode.

# Right now, with just nn.Linear, it doesn't change much. 
# Later when we introduce things like dropout and batch normalization, this becomes very important.

# model.eval() switches the model into evaluation mode.

# torch.no_grad() tells PyTorch I am only making predictions and not calculating gradients


In [20]:
# TRAINING SETUP AS OF THIS POINT
# batch
#  ↓
# move to GPU
#  ↓
# prediction
#  ↓
# loss
#  ↓
# backpropagation
#  ↓
# optimizer update

# VALIDATION
# batch
#  ↓
# move to GPU
#  ↓
# prediction
#  ↓
# loss
#  ↓
# NO backpropagation
#  ↓
# NO parameter update

In [21]:
# inspect learned parameters
for name, param in model.named_parameters():
    print(name, param.data)

linear.weight tensor([[3.0146]], device='cuda:0')
linear.bias tensor([1.0023], device='cuda:0')


In [23]:
# With unscaled inputs ranging from 1–100, 
# SGD rapidly learned the slope but the bias converged much more slowly. 
# Increasing the number of epochs improved the bias only gradually, 
# illustrating how feature scale can affect optimization.

## Adam Optimizer Comparison

In [24]:
model = LinearModel().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

In [25]:
epochs = 500

for epoch in range(epochs):

    model.train()
    train_loss = 0.0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        predictions = model(xb)

        loss = criterion(predictions, yb)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            predictions = model(xb)

            loss = criterion(predictions, yb)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    if epoch % 50 == 0:
        print(
            f"Epoch {epoch:4d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f}"
        )

Epoch    0 | Train Loss: 48719.3918 | Val Loss: 40110.5469
Epoch   50 | Train Loss: 2715.5585 | Val Loss: 2189.2701
Epoch  100 | Train Loss: 23.0671 | Val Loss: 17.8576
Epoch  150 | Train Loss: 1.2024 | Val Loss: 1.4463
Epoch  200 | Train Loss: 1.1399 | Val Loss: 1.3986
Epoch  250 | Train Loss: 1.0837 | Val Loss: 1.3300
Epoch  300 | Train Loss: 1.0169 | Val Loss: 1.2479
Epoch  350 | Train Loss: 0.9390 | Val Loss: 1.1516
Epoch  400 | Train Loss: 0.8514 | Val Loss: 1.0429
Epoch  450 | Train Loss: 0.7542 | Val Loss: 0.9242


In [26]:
for name, param in model.named_parameters():
    print(name, param.data)

linear.weight tensor([[2.9753]], device='cuda:0')
linear.bias tensor([3.6618], device='cuda:0')


In [32]:
model = LinearModel().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [33]:
epochs = 1500

for epoch in range(epochs):

    model.train()
    train_loss = 0.0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        predictions = model(xb)

        loss = criterion(predictions, yb)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            predictions = model(xb)

            loss = criterion(predictions, yb)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    if epoch % 50 == 0:
        print(
            f"Epoch {epoch:4d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f}"
        )

Epoch    0 | Train Loss: 39829.4685 | Val Loss: 33529.3027
Epoch   50 | Train Loss: 31264.7219 | Val Loss: 26321.0859
Epoch  100 | Train Loss: 24160.1970 | Val Loss: 20322.3438
Epoch  150 | Train Loss: 18294.5040 | Val Loss: 15391.3828
Epoch  200 | Train Loss: 13519.3310 | Val Loss: 11371.4648
Epoch  250 | Train Loss: 9696.3724 | Val Loss: 8148.6262
Epoch  300 | Train Loss: 6698.1273 | Val Loss: 5627.6909
Epoch  350 | Train Loss: 4420.6497 | Val Loss: 3708.9762
Epoch  400 | Train Loss: 2752.8062 | Val Loss: 2307.9097
Epoch  450 | Train Loss: 1594.7556 | Val Loss: 1335.4740
Epoch  500 | Train Loss: 843.2142 | Val Loss: 704.5318
Epoch  550 | Train Loss: 396.8424 | Val Loss: 330.5212
Epoch  600 | Train Loss: 161.0904 | Val Loss: 133.5862
Epoch  650 | Train Loss: 54.2099 | Val Loss: 44.6810
Epoch  700 | Train Loss: 14.5207 | Val Loss: 11.8654
Epoch  750 | Train Loss: 3.0293 | Val Loss: 2.4618
Epoch  800 | Train Loss: 0.5918 | Val Loss: 0.5248
Epoch  850 | Train Loss: 0.2376 | Val Loss: 0.2

In [34]:
for name, param in model.named_parameters():
    print(name, param.data)

linear.weight tensor([[2.9904]], device='cuda:0')
linear.bias tensor([2.6458], device='cuda:0')
